In [0]:
# Installation des librairies de base
%pip install langgraph langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Mise à jour forcée pour éviter le conflit Pydantic/Typing (l'erreur de tout à l'heure)
%pip install -U typing_extensions pydantic langchain-core

# Redémarrage du moteur pour prendre en compte les installations
dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os

# 1. Configuration et imports
sys.path.append(os.path.abspath('..'))
from src.interview_graph import build_interview_graph
try:
    hf_token = dbutils.secrets.get(scope="llm_secrets", key="huggingface_token")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    print(" Token HuggingFace récupéré et injecté dans l'environnement !")
except Exception as e:
    print(f" Erreur avec le secret Databricks : {e}")
    raise

# 2. Paramètres de l'entretien
sujet = "SQL et Bases de Données"
nb_questions = 3

print(f"\n Initialisation de l'entretien sur : {sujet}")
app = build_interview_graph()

# --- PHASE 1 : GÉNÉRATION DES QUESTIONS ---
etat_initial = {
    "sujet_entretien": sujet,
    "nombre_questions": nb_questions,
    "questions": [],
    "reponses": [],
    "contextes_rag": [],
    "feedback_recruteur": "",
    "passed_recruteur": False,
    "verdict_juge": "",
    "juge_est_daccord": False,  
    "iteration": 0,
    "historique_juge": [], 
    "notes_juge_independant": "",
    "ecart_total": 0,
}

config = {"recursion_limit": 10}

try:
    resultat_gen = app.invoke(etat_initial, config=config)
except Exception as e:
    print(f" Échec de la génération des questions : {e}")
    raise

questions_posees = resultat_gen.get("questions", [])

# Validation : vérifier qu'on a bien le bon nombre de questions
if len(questions_posees) == 0:
    raise ValueError("Aucune question générée par l'Agent 1.")
if len(questions_posees) != nb_questions:
    print(f" Attention : {len(questions_posees)} questions générées au lieu de {nb_questions}")

# --- PHASE 2 : INTERACTION LIVE ---
print("\n" + "=" * 60)
print("  L'ENTRETIEN COMMENCE MAINTENANT")
print("=" * 60 + "\n")

reponses_utilisateur = []
for i, q in enumerate(questions_posees):
    print(f" QUESTION {i+1} : {q}")
    reponse = input(f"   Votre réponse à la Q{i+1} > ").strip()
    if not reponse:
        reponse = "(pas de réponse)"
    reponses_utilisateur.append(reponse)
    print(f"    Réponse enregistrée.\n")

# --- PHASE 3 : ÉVALUATION GLOBALE ---
print("  Analyse de vos réponses par le Recruteur et le Juge...\n")

etat_final = {
    **resultat_gen,
    "reponses": reponses_utilisateur,
    "juge_est_daccord": False,   # ← FIX : reset à False avant éval
}

try:
    verdict_final = app.invoke(etat_final, config=config)
except Exception as e:
    print(f" Erreur durant l'évaluation : {e}")
    raise

# --- AFFICHAGE DES RÉSULTATS ---
print("\n" + "=" * 60)
print("  RÉSULTAT DE L'ENTRETIEN")
print("=" * 60)

# Affichage des contextes RAG utilisés (debug / transparence)
print("\n  THÉORIE DE RÉFÉRENCE UTILISÉE :")
for i, ctx in enumerate(verdict_final.get("contextes_rag", [])):
    print(f"  Q{i+1} → {ctx[:120]}...")

print(f"\n  BILAN DU RECRUTEUR :")
print(verdict_final.get("feedback_recruteur", "(aucun bilan)"))

decision = verdict_final.get("passed_recruteur", False)
print(f"\n  DÉCISION DU RECRUTEUR : {' ADMIS' if decision else ' RECALÉ'}")

print(f"\n  AVIS DU JUGE :")
print(verdict_final.get("verdict_juge", "(aucun avis)"))

juge_ok = verdict_final.get("juge_est_daccord", False)
print(f"\n  SUPERVISION : {' VALIDÉE par le Juge' if juge_ok else ' REJETÉE par le Juge'}")
print(f"\n NOMBRE D'ITÉRATIONS : {verdict_final.get('iteration', 1)}")
historique = verdict_final.get("historique_juge", [])
if historique:
    print(f"\n HISTORIQUE DES REJETS DU JUGE :")
    for h in historique:
        print(f"  - {h}")